In [ ]:
# Multi-Head Attention 
# Implemented using a single class with weight splits 

from re import I
import torch 
import torch.nn as nn 

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in: int, d_out: int, context_length: int, num_heads: int, dropout: float, qkv_bias: bool = False):
        super().__init__()
        self.d_out = d_out 
        self.num_heads = num_heads
        self.dropout = dropout

        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Attention Weights with Value 
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout) 

        # Casual Mask
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      batch_size, num_tokens, d_in = x.shape

      queries = self.W_q(x)
      keys = self.W_k(x)
      values = self.W_v(x)

      # Split output dimensions into large Q, K, V tensor matrices 
      queries = queries.view(batch_size, num_tokens, self.num_heads, self.d_out)
      keys = keys.view(batch_size, num_tokens, self.num_heads, self.d_out)
      values = values.view(batch_size, num_tokens, self.num_heads, self.d_out)

      # Transpose 
      queries = queries.transpose(1, 2)
      keys = keys.transpose(1, 2)
      values = values.transpose(1, 2)

      # Compute attention scores per head 
      attn_scores = queries @ keys.transpose(2, 3) 

      # Apply casual mask
      mask = self.mask[:num_tokens, :num_tokens]
      attn_scores = attn_scores.masked_fill(mask, float("-inf")) 

      # Normalize through software to obtain attention weights 
      attn_weights = torch.softmax(attn_scores / (self.head_dim ** 0.5), dim=-1)
      attn_weights = self.dropout(attn_weights)

      # Weighted sum of values 
      context_vector = attn_weights @ values 
      context_vec = context_vec.transpose(1, 2)
      context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

      # Project to output dimension 
      context_vec = self.out_proj(context_vec)

      return context_vec 
        